In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("alsaniipe/cebradata")

print("Path to dataset files:", path)

100%|██████████| 1.32G/1.32G [00:17<00:00, 82.2MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/alsaniipe/cebradata/versions/1


In [ ]:
!ls /root/.cache/kagglehub/datasets/alsaniipe/cebradata/versions/1/celeb_dataset

images


In [ ]:
!pip install Pillow
!pip install torchvision
!pip install torch
!pip install numpy

In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import torch
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler, DDPMPipeline
from diffusers.optimization import get_cosine_schedule_with_warmup
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import os
from PIL import Image
import bitsandbytes as bnb



# Kayıt yollarını Drive olarak ayarla
DRIVE_PATH = "/content/drive/MyDrive/t4_faces_project"
MODEL_SAVE_PATH = f"{DRIVE_PATH}/model_checkpoint"
SAMPLE_SAVE_PATH = f"{DRIVE_PATH}/face_samples"

os.makedirs(SAMPLE_SAVE_PATH, exist_ok=True)

# 2. CİHAZ VE AYARLAR
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

image_dir = "/root/.cache/kagglehub/datasets/alsaniipe/cebradata/versions/1/celeb_dataset/images"
res = 64
batch_size = 32

transform = transforms.Compose([
    transforms.Resize((res, res)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

class FaceDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg') or f.endswith('.png')]
        self.transform = transform
    def __len__(self):
        return len(self.image_files)
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image

dataset = FaceDataset(image_dir, transform=transform)
train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)

# 3. MİMARİ VE DERLEME
model = UNet2DModel(
    sample_size=res,
    in_channels=3,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(128, 256, 256, 512),
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D", # İsim doğru: Down kısmında DownBloğu
        "DownBlock2D"
    ),
    up_block_types=(
        "UpBlock2D",
        "AttnUpBlock2D",   # BURAYI DÜZELTTİK: Up kısmında UpBloğu olmalı
        "UpBlock2D",
        "UpBlock2D"
    ),
).to(device)
# 4. OPTİMİZASYON
optimizer = bnb.optim.Adam8bit(model.parameters(), lr=1e-4)
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
scaler = torch.amp.GradScaler('cuda')

epochs = 30
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer, num_warmup_steps=500,
    num_training_steps=(len(train_dataloader) * epochs),
)

# 5. EĞİTİM DÖNGÜSÜ
print(f"🚀 Eğitim Başladı. Görseller ve Model Google Drive'a ({DRIVE_PATH}) kaydedilecek.")

for epoch in range(epochs):
    model.train()
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch}")

    for step, images in enumerate(progress_bar):
        images = images.to(device, non_blocking=True)
        noise = torch.randn_like(images).to(device)
        timesteps = torch.randint(0, 1000, (images.shape[0],), device=device).long()
        noisy_images = noise_scheduler.add_noise(images, noise, timesteps)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            noise_pred = model(noisy_images, timesteps).sample
            loss = F.mse_loss(noise_pred, noise)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()
        progress_bar.set_postfix(loss=loss.item())

    # --- HER EPOCH SONU: GÖRSELLEŞTİRME ---
    model.eval()
    # torch.compile kullanıldıysa asıl modeli unwrap yapıyoruz
    actual_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    pipeline = DDPMPipeline(unet=actual_model, scheduler=noise_scheduler)

    with torch.amp.autocast('cuda'):
        # Her epoch sonu 4 adet örnek üret
        gen_images = pipeline(batch_size=4, num_inference_steps=30).images

    for i, img in enumerate(gen_images):
        img.save(f"{SAMPLE_SAVE_PATH}/epoch_{epoch}_sample_{i}.png")

    # --- HER EPOCH SONU: DRIVE YEDEKLEME ---
    actual_model.save_pretrained(MODEL_SAVE_PATH)
    print(f"✅ Epoch {epoch} bitti. Görseller ve Model Drive'a yedeklendi.")

print("🏁 Tüm eğitim tamamlandı!")

🚀 Eğitim Başladı. Görseller ve Model Google Drive'a (/content/drive/MyDrive/t4_faces_project) kaydedilecek.


Epoch 0:   0%|          | 0/6331 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

✅ Epoch 0 bitti. Görseller ve Model Drive'a yedeklendi.


Epoch 1:   0%|          | 0/6331 [00:00<?, ?it/s]

...

✅ Epoch 4 bitti. Görseller ve Model Drive'a yedeklendi.


Epoch 5:   0%|          | 0/6331 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

✅ Epoch 5 bitti. Görseller ve Model Drive'a yedeklendi.


Epoch 6:   0%|          | 0/6331 [00:00<?, ?it/s]

In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00


In [ ]:
import os, torch, gradio as gr
from diffusers import UNet2DModel, DDPMScheduler, DDPMPipeline
from transformers import Swin2SRForImageSuperResolution, Swin2SRImageProcessor
from safetensors.torch import load_file
from PIL import Image
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. 11. EPOCH YOLUNU TANIMLA
# Not: Eğer klasör adın farklıysa (örn: checkpoint-11) burayı ona göre güncelle
weights_path_11 = "/content/drive/MyDrive/t4_faces_project/checkpoints/checkpoint_epoch_10/diffusion_pytorch_model.safetensors"
# Eğer ana klasöre kaydettiysen yedek yol:
backup_weights_path = "/content/drive/MyDrive/t4_faces_project/model_checkpoint/diffusion_pytorch_model.safetensors"

# 2. MODEL MİMARİSİ (Aynı kalsın)
model_config = {
    "sample_size": 64,
    "in_channels": 3,
    "out_channels": 3,
    "layers_per_block": 2,
    "block_out_channels": [128, 256, 256, 512],
    "down_block_types": ["DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "DownBlock2D"],
    "up_block_types": ["UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"]
}

model = UNet2DModel(**model_config).to(device)

# 3. 11. EPOCH AĞIRLIKLARINI YÜKLE
if os.path.exists(weights_path_11):
    state_dict = load_file(weights_path_11)
    model.load_state_dict(state_dict)
    print("🎯 Başarı! 11. Epoch ağırlıkları yüklendi.")
elif os.path.exists(backup_weights_path):
    state_dict = load_file(backup_weights_path)
    model.load_state_dict(state_dict)
    print("⚠️ 11. Epoch klasörü bulunamadı, ana klasördeki güncel ağırlıklar yüklendi.")
else:
    print("❌ HATA: Ağırlık dosyası bulunamadı! Lütfen Drive yolunu kontrol et.")

# 4. PIPELINE VE UPSCALER
scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule='linear')
pipeline = DDPMPipeline(unet=model, scheduler=scheduler).to(device)
processor = Swin2SRImageProcessor.from_pretrained("caidas/swin2SR-classical-sr-x4-64")
upscaler = Swin2SRForImageSuperResolution.from_pretrained("caidas/swin2SR-classical-sr-x4-64").to(device)

def generate_face(inference_steps):
    low_res = pipeline(batch_size=1, num_inference_steps=int(inference_steps)).images[0]
    inputs = processor(low_res, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = upscaler(**inputs)
    output_tensor = outputs.reconstruction.squeeze().cpu().clamp(0, 1)
    output_array = (output_tensor.numpy().transpose(1, 2, 0) * 255.0).round().astype(np.uint8)
    high_res = Image.fromarray(output_array)
    return low_res, high_res

# 5. ARAYÜZ
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 👤 AI Face Generator")
    with gr.Row():
        steps = gr.Slider(label="Netlik (Steps)", minimum=30, maximum=100, value=50, step=1)
        btn = gr.Button("🎨 Yüz Oluştur", variant="primary")
    with gr.Row():
        out_low = gr.Image(label="Ham Model Çıktısı")
        out_high = gr.Image(label="AI Netleştirilmiş")
    btn.click(fn=generate_face, inputs=steps, outputs=[out_low, out_high])

demo.launch(share=True, debug=True)

⚠️ 11. Epoch klasörü bulunamadı, ana klasördeki güncel ağırlıklar yüklendi.


Loading weights:   0%|          | 0/726 [00:00<?, ?it/s]

/tmp/ipykernel_3815/2644222620.py:58: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c2c46beed4e09fe1a7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]